# AR Cloud Multi-Agent RAG System

This notebook demonstrates a multi-agent RAG system with department routing, ChromaDB, OpenAI, and Langfuse.

## 1. Setup and Imports

In [1]:
from src.config import CHROMA_PERSIST_DIRECTORY
from src.retrieval.chunking import DEPARTMENT_FOLDERS
from src.retrieval.retriever import retrieve_documents
from src.router import route_question
from src.agents.department_agent import answer_department_question
from src.orchestrator import process_question

print('Chroma directory:', CHROMA_PERSIST_DIRECTORY)

Chroma directory: /Users/cristianarielachetoni/code/AI Engeenering/M3/department-rag-router/storage/chroma


## 2. Document Collections and Vector Stores

Each department has an independent ChromaDB collection. Run the indexing command before executing retrieval examples.

In [2]:
for department in DEPARTMENT_FOLDERS:
    print(department, f'{department}_documents')

hr hr_documents
technology technology_documents
finance finance_documents


## 3. Specialized Department Agents

In [3]:
question = 'How do I request annual leave?'
documents = retrieve_documents(question, 'hr', k=3)
print('Retrieved chunks:', len(documents))
print('Document IDs:', [doc.metadata['document_id'] for doc in documents])

Retrieved chunks: 3
Document IDs: ['HR-002', 'HR-002', 'HR-001']


### Department Agent Examples

The system uses three specialized RAG agents: Human Resources, Technology, and Finance. Each agent retrieves information from its own ChromaDB collection and generates answers grounded in department-specific documents.

The following examples demonstrate the behavior of each specialized agent.

In [4]:

department_questions = {
    "hr": "How do I request annual leave?",
    "technology": "How do I report a technical incident?",
    "finance": "How do I submit a reimbursement request?",
}

for department, question in department_questions.items():
    result = answer_department_question(
        question=question,
        department=department,
        k=3,
    )

    print(f"\nDepartment: {department.upper()}")
    print(f"Question: {question}")
    print(f"Answer: {result['answer']}")
    print("Sources:")
    for source in result["sources"]:
        print(f"  - {source['document_id']}: {source['source']}")
    print("-" * 60)



Department: HR
Question: How do I request annual leave?
Answer: To request annual leave at AR Cloud:

1. Submit your vacation request through the employee portal.
2. Include:
   - The first day of absence
   - The last day of absence
   - The colleague responsible for urgent handover items
3. Submit the request at least 10 working days before your first day of vacation.
4. You should discuss the planned absence with your manager before submitting, especially if it overlaps with a product release or another team member’s approved absence.
Sources:
  - HR-002: data/hr_docs/leave_policy.md
  - HR-002: data/hr_docs/leave_policy.md
  - HR-001: data/hr_docs/vacation_policy.md
------------------------------------------------------------

Department: TECHNOLOGY
Question: How do I report a technical incident?
Answer: To report a technical incident, create a recorded support request/ticket through the official support process. An informal chat to an individual technician does **not** replace a 

## 4. Router and Orchestrator

In [5]:
routing = route_question(question)
print('Department:', routing.department)
print('Reason:', routing.reason)

Department: finance
Reason: Reimbursement requests are handled by finance as part of expenses and payments procedures.


In [6]:
result = process_question(question)
print('Answer:', result['answer'])
print('Sources:', result['sources'])

Answer: To submit a reimbursement request, use AR Cloud’s approved submission process.

Your request should include:

- Employee name
- Department
- Expense date
- Expense category
- Amount
- Currency
- Business justification

You must also attach or make available supporting documentation through the approved internal system. Acceptable documentation may include receipts, invoices, electronic payment confirmations, or other Finance-approved records.

The documentation should show the vendor, transaction date, amount paid, currency, and goods or services purchased. If the expense relates to a project, trip, or activity, include that explanation as well.

Before submitting, check that the request matches the supporting documents. Requests with incorrect amounts, missing receipts, or unclear descriptions may be returned for correction.

Employees should submit reimbursement requests within 15 calendar days after the expense is incurred or after the related business trip is completed.
Sou

## 5. Tests and Examples

In [7]:
from pathlib import Path
from src.evaluation import load_test_queries

queries = load_test_queries(Path('test_queries.json'))
print('Available evaluation queries:', len(queries))
for query in queries[:3]:
    print(query['id'], query['question'])

Available evaluation queries: 15
HR-01 How do I request annual vacation leave?
HR-02 How can I request a recurring remote work arrangement?
HR-03 Who is responsible for mandatory employee training?


### Evaluation Results

The evaluation dataset contains 15 queries covering the three departments and edge cases.

The evaluation measures routing accuracy, retrieval hit rate, and answer quality using an LLM judge. The quality score ranges from 0 to 10.

The following cell loads the previously generated evaluation report without making additional OpenAI API calls.

In [8]:

import json
from pathlib import Path

report_path = Path("evaluation/report.json")

with report_path.open(encoding="utf-8") as file:
    report = json.load(file)

metrics = report["metrics"]

print("Evaluation Summary")
print("-" * 40)
print(f"Total queries: {metrics['total_queries']}")
print(f"Routing accuracy: {metrics['routing_accuracy']:.2%}")
print(f"Retrieval hit rate: {metrics['retrieval_hit_rate']:.2%}")
print(
    "Average answer quality: "
    f"{metrics['average_answer_quality']:.2f}/10"
)
print(f"Quality evaluations: {metrics['quality_evaluated']}")


Evaluation Summary
----------------------------------------
Total queries: 15
Routing accuracy: 100.00%
Retrieval hit rate: 100.00%
Average answer quality: 9.71/10
Quality evaluations: 14


In [9]:

print(f"{'Query':<12} {'Routing':<12} {'Retrieval':<12} {'Quality'}")
print("-" * 50)

for result in report["results"]:
    routing = result["routing_correct"]
    retrieval = result["retrieval_hit"]
    quality = result.get("answer_quality_score")

    routing_text = "N/A" if routing is None else str(routing)
    retrieval_text = "N/A" if retrieval is None else str(retrieval)
    quality_text = "N/A" if quality is None else f"{quality}/10"

    print(
        f"{result['id']:<12} "
        f"{routing_text:<12} "
        f"{retrieval_text:<12} "
        f"{quality_text}"
    )


Query        Routing      Retrieval    Quality
--------------------------------------------------
HR-01        True         True         10/10
HR-02        True         True         10/10
HR-03        True         True         10/10
HR-04        True         True         9/10
TECH-01      True         True         10/10
TECH-02      True         True         10/10
TECH-03      True         True         9/10
TECH-04      True         True         10/10
FIN-01       True         True         9/10
FIN-02       True         True         10/10
FIN-03       True         True         10/10
FIN-04       True         True         10/10
EDGE-01      True         True         10/10
EDGE-02      True         True         9/10
EDGE-03      N/A          N/A          N/A


## 6. Langfuse Observability

The orchestrator records routing, retrieval, and generation traces in Langfuse.

Run the orchestration cells above and inspect the resulting traces in the configured Langfuse project.

### Trace Inspection and LLM Judge Scores

Langfuse records the execution of the multi-agent RAG workflow, including:

- Department routing and LLM generation.
- Document retrieval from the department-specific vector store.
- Input questions, generated answers, and execution metadata.
- Model usage, latency, and estimated costs.
- LLM judge scores associated with the corresponding traces.

The evaluation pipeline uses the Langfuse Score API to record answer quality scores from 0 to 10.

To inspect the results, open the configured Langfuse project and navigate to **Tracing** for execution details or **Scores** for answer quality evaluations.